## Manipulación de Datos NASA 
- Se convierten los datos Diarios a semanales, se escogen los principales departamentos (1 municipio por departamento) con mayor producción de Café

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [11]:
# -------------------------------------------------------------------
# 1. Cargar todos los CSV diarios (uno por departamento)
# -------------------------------------------------------------------
data_path = Path(r"C:\Users\junio\OneDrive\Escritorio\Andes\MIID\Gerencia Proyectos\Anteproyecto\data\NASA\Base_principal")

files = list(data_path.glob("POWER_Point_Daily_*.csv"))

print("Carpeta usada:", data_path)
print("Número de archivos encontrados:", len(files))
if len(files) == 0:
    raise FileNotFoundError(
        f"No se encontraron archivos 'POWER_Point_Daily_*.csv' en {data_path}"
    )


Carpeta usada: C:\Users\junio\OneDrive\Escritorio\Andes\MIID\Gerencia Proyectos\Anteproyecto\data\NASA\Base_principal
Número de archivos encontrados: 5


In [12]:
dfs = []
for f in files:
    # Ej: POWER_Point_Daily_20210101_20251031_ANTIOQUIA_LST.csv
    name = f.stem
    dept = name.split("_")[-2]  # ANTIOQUIA, CALDAS, CAUCA, HUILA, TOLIMA

    df = pd.read_csv(f, sep=';')
    df["DEPARTAMENTO"] = dept
    dfs.append(df)

# Unimos todos los departamentos
data = pd.concat(dfs, ignore_index=True)

In [13]:
# ---------------------------------------------------------
# ---------------------------------------------------------
# 2. Normalizar nombres y construir fecha YEAR + DOY
# ---------------------------------------------------------
data.columns = data.columns.str.strip().str.upper()
print("Columnas después de normalizar:")
print(data.columns.tolist())

data["YEAR"] = data["YEAR"].astype(int)
data["DOY"] = data["DOY"].astype(int)

data["date"] = (
    pd.to_datetime(data["YEAR"], format="%Y") +
    pd.to_timedelta(data["DOY"] - 1, unit="D")     # DOY 1–365/366
)


Columnas después de normalizar:
['YEAR', 'DOY', 'PRECTOTCORR', 'IMERG_PRECTOT', 'T2M', 'T2M_MAX', 'T2M_MIN', 'RH2M', 'QV2M', 'GWETTOP', 'GWETROOT', 'ALLSKY_SFC_SW_DWN', 'WS2M', 'DEPARTAMENTO']


In [14]:
# ---------------------------------------------------------
# 3. Limpiar valores faltantes (-999, -9999)
# ---------------------------------------------------------
bad_values = [-999.0, -9999.0]
num_cols = data.select_dtypes(include="number").columns
data[num_cols] = data[num_cols].replace(bad_values, np.nan)


In [15]:
# ---------------------------------------------------------
# 4. Agregación semanal por departamento
# ---------------------------------------------------------
data = data.set_index("date")
data = data.drop(columns=["YEAR", "DOY"], errors="ignore")

weekly_dept = (
    data
    .groupby("DEPARTAMENTO")
    .resample("W")      # semana (termina domingo). Si quieres ISO: "W-MON"
    .mean(numeric_only=True)
    .reset_index()
)


In [16]:
# ---------------------------------------------------------
# 5. Proxy nacional (promedio de los 5 deptos)
# ---------------------------------------------------------
weekly_national = (
    weekly_dept
    .groupby("date")
    .mean(numeric_only=True)
    .reset_index()
)
weekly_national.insert(1, "DEPARTAMENTO", "NACIONAL")


In [17]:
# ---------------------------------------------------------
# 6. Dejar solo las variables de interes
# ---------------------------------------------------------
vars_keep = [
    "date", "DEPARTAMENTO",
    "PRECTOTCORR", "IMERG_PRECTOT",
    "T2M", "T2M_MAX", "T2M_MIN",
    "RH2M", "QV2M",
    "GWETTOP", "GWETROOT",
    "ALLSKY_SFC_SW_DWN", "WS2M"
]

weekly_dept = weekly_dept[vars_keep]
weekly_national = weekly_national[vars_keep]

In [20]:
# ---------------------------------------------------------
# 7. Guardar archivos
# ---------------------------------------------------------
weekly_dept.to_csv(data_path / "VARIABLES_NASA_DEPARTAMENTAL_SEMANAL.csv", index=False)
weekly_national.to_csv(data_path / "VARIABLES_NASA_CONSOLIDADO_NAL.csv", index=False)

# Consolidado (deptos + nacional)
consolidado = pd.concat([weekly_dept, weekly_national], ignore_index=True)
consolidado.to_csv(data_path / "VARIABLES_NASA_CONSOLIDADOo.csv", index=False)

print("Se generaron los tres archivos en:", data_path)

Se generaron los tres archivos en: C:\Users\junio\OneDrive\Escritorio\Andes\MIID\Gerencia Proyectos\Anteproyecto\data\NASA\Base_principal
